# X (Twitter) Trend Lifespan & Viral Prediction AI (Mamba Edition)

This notebook provides a complete pipeline for analyzing **X (formerly Twitter)** trends, predicting viral lifespan using a **Pure PyTorch Mamba (State Space Model)**, and generating AI summaries.

**Note on Data Access:**
X (Twitter) restricts automated access. This notebook attempts to use `ntscraper` (if installed). If unavailble or blocked, it uses a **simulation mode** to generate realistic data based on the keyword, ensuring you can still run and verify the Mamba model architecture.

**Instructions:**
1. Run the **Setup & Imports** cell.
2. Run the **Mamba Model & Core Functions** cell.
3. Run the **Execution** cell.

In [25]:
# --- CELL 1: SETUP & IMPORTS ---
import pandas as pd
import time
import datetime
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from IPython.display import display, Markdown
import warnings
import re

# Try importing ntscraper for real scraping (user must install it: pip install ntscraper)
try:
    from ntscraper import Nitter
    HAS_NTSCRAPER = True
except ImportError:
    HAS_NTSCRAPER = False

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")
if HAS_NTSCRAPER:
    print("ntscraper detected. Will attempt real scraping.")
else:
    print("ntscraper not found. Using Simulation Mode (generating realistic mock data).")

Imports loaded successfully.
ntscraper not found. Using Simulation Mode (generating realistic mock data).


In [26]:
# --- CELL 2: CORE FUNCTIONS & MODELS (MAMBA ENABLED) ---

# 1. Minimal Mamba Layer (Pure PyTorch Implementation)
class MinimalMambaLayer(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.dt_rank = object = int(d_model / 16)
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )

        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        self.A_log = nn.Parameter(torch.log(torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        batch, seq_len, _ = x.shape
        x_and_res = self.in_proj(x)  # [batch, seq, 2*inner]
        (x, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)

        x = x.transpose(1, 2)
        x = self.conv1d(x)[:, :, :seq_len]
        x = x.transpose(1, 2)

        x = F.silu(x)
        y = self.ssm_scan(x)
        y = y * F.silu(res)
        output = self.out_proj(y)
        return output

    def ssm_scan(self, x):
        # Simplified Selective Scan
        batch, seq_len, d_inner = x.shape
        delta_BC = self.x_proj(x)
        delta, B, C = torch.split(delta_BC, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        A = -torch.exp(self.A_log)
        ys = []
        h = torch.zeros(batch, d_inner, self.d_state, device=x.device)
        for t in range(seq_len):
            dt = delta[:, t, :].unsqueeze(-1)
            dA = torch.exp(A * dt)
            dB = (dt * B[:, t, :].unsqueeze(1))
            xt = x[:, t, :].unsqueeze(-1)
            h = h * dA + dB * xt
            y_t = torch.sum(h * C[:, t, :].unsqueeze(1), dim=-1)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return y + x * self.D

# 2. Temporal Model using Mamba
class TemporalSSM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = MinimalMambaLayer(d_model=dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        out = self.mamba(x)
        return self.norm(out)

class LifespanPredictor(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.ReLU()
        )
    def forward(self, x):
        return self.model(x)

# 3. Twitter Scraper (With IMPACTFUL Fallback Simulator)
def fetch_twitter_posts(keyword, pages=5):
    all_posts = []
    
    # Attempt Real Scraping
    if HAS_NTSCRAPER:
        try:
            scraper = Nitter()
            print(f"Using ntscraper to find posts for '{keyword}'...")
            tweets = scraper.get_tweets(keyword, mode='term', number=20)
            if tweets and 'tweets' in tweets:
                for t in tweets['tweets']:
                    all_posts.append({
                        "post_id": t['link'].split('/')[-1] if 'link' in t else str(random.randint(1000,9999)),
                        "text": t['text'],
                        "upvotes": t['stats']['likes'],
                        "comments": t['stats']['comments'],
                        "reposts": t['stats'].get('retweets', 0), 
                        "created_utc": datetime.datetime.strptime(t['date'], "%b %d, %Y · %I:%M %p UTC").timestamp(),
                        "platform": "X",
                        "keyword": keyword
                    })
        except Exception as e:
            print(f"Nitter scraping failed ({e}). Switching to simulation.")
    
    # Fallback / Simulation Mode (Improved Coherence & Quantity)
    if not all_posts:
        print(f"Generating COHERENT simulated data for '{keyword}'...")
        base_time = time.time()
        current_year = datetime.datetime.now().year
        
        templates = [
            f"BREAKING: Major new developments reported regarding {keyword} this morning.",
            f"Analysts are predicting a significant shift in strategy for {keyword} in {current_year}.",
            f"The internet is debating the latest statement about {keyword}.",
            f"Thread: Here is everything you need to know about the {keyword} situation 🧵",
            f"Video surfacing of {keyword} is currently trending worldwide.",
            f"Updates on {keyword}: Sources confirm new timeline for upcoming events.",
            f"Why {keyword} remains a top priority topic for voters/consumers this week.",
            f"Top story: The impact of {keyword} on the current market landscape.",
            f"Opinion: The recent {keyword} updates could change everything we know.",
            f"Live coverage: Tracking the latest updates on the {keyword} phenomenon."
        ]
        
        for i in range(60):
            txt = random.choice(templates)
            is_viral = i < 10
            likes = random.randint(10000, 50000) if is_viral else random.randint(10, 500)
            
            all_posts.append({
                "post_id": str(100000 + i),
                "text": txt,
                "upvotes": likes,
                "comments": int(likes * 0.1),
                "reposts": int(likes * 0.25),
                "created_utc": base_time - random.randint(0, 86400 * 5),
                "platform": "X",
                "keyword": keyword
            })
            
    return pd.DataFrame(all_posts)

# 4. Semantic Filtering
def filter_by_keyword_relevance(df, keyword, embedder, threshold=0.35):
    keyword_emb = embedder.encode(keyword).reshape(1, -1)
    similarities = []
    for emb in df["embedding"]:
        sim = cosine_similarity(emb.reshape(1, -1), keyword_emb)[0][0]
        similarities.append(sim)
    df = df.copy()
    df["similarity"] = similarities
    return df[df["similarity"] >= threshold]

# 5. Data Aggregation
def aggregate_daily(df, embedder):
    if df.empty: return None, None
    df["date"] = df["created_utc"].apply(lambda x: datetime.datetime.fromtimestamp(x).date())
    if "embedding" not in df.columns:
         df["embedding"] = df["text"].apply(lambda x: embedder.encode(x))
    
    daily_vectors = []
    daily_meta = []
    for date, g in df.groupby("date"):
        text_emb = np.mean(np.vstack(g["embedding"]), axis=0)
        engagement = np.array([g["upvotes"].mean(), g["comments"].mean()])
        vector = np.concatenate([text_emb, engagement])
        daily_vectors.append(vector)
        daily_meta.append({"date": date, "posts": len(g)})
    if not daily_vectors: return None, None
    X = np.vstack(daily_vectors)
    return X, daily_meta

# 6. Utilities
def summarize_trend(df):
    # Twitter style summary
    top = df.sort_values("upvotes", ascending=False).head(5)
    bullets = []
    for t in top["text"]:
        bullets.append("• " + t)
    return "\n".join(bullets)

def format_twitter_link(post_id, keyword):
    if len(post_id) > 10:
        return f"https://x.com/user/status/{post_id}"
    return f"https://x.com/search?q={keyword}"

def get_trending_posts_display(df, keyword):
    top = df.sort_values("upvotes", ascending=False).head(5)
    links = []
    for i, row in top.iterrows():
        title = row['text'][:100].replace("[", "(").replace("]", ")") + ("..." if len(row['text']) > 100 else "")
        url = format_twitter_link(row['post_id'], keyword)
        # Enhanced Stats Display
        stats = f" (❤️ {row['upvotes']} | 💬 {row['comments']} | 🔁 {row.get('reposts', 0)})"
        links.append(f"- [{title}]({url}){stats}")
    return "\n".join(links)

def predict_viral_days(X, encoder, head):
    x_seq = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        h = encoder(x_seq)
        final_state = h[:, -1, :]
        pred = head(x_seq)[:, -1, :]
    val = int(pred.item())
    return max(1, abs(val) % 30)

def get_llm_summary(df):
    # Gather more context (Top 20 posts) to allow for longer summaries
    top_posts = df.sort_values("upvotes", ascending=False).head(20)["text"].tolist()
    unique_text = list(set(top_posts))
    combined_text = " ".join(unique_text)
    
    try:
        # Increased max_length and min_length for longer output (approx 5 lines)
        summarizer = pipeline("summarization", model="Falconsai/text_summarization", framework="pt")
        input_text = "summarize: " + combined_text
        summary = summarizer(input_text, max_length=150, min_length=60, do_sample=False)[0]['summary_text']
    except Exception as e:
        summary = combined_text[:300] + "... (Summary unavailable due to model limit)"
    return summary

print("Mamba Model (X Edition) initialized.")

Mamba Model (X Edition) initialized.


In [27]:
# --- CELL 3: MAIN PIPELINE ---

def run_twitter_trend_demo(keyword):
    print(f"\nAnalyzing X/Twitter Trend for: '{keyword}'")
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    
    print("Fetching Tweets...")
    df = fetch_twitter_posts(keyword)
    
    print("Processing embedding vectors...")
    df['embedding'] = df['text'].apply(lambda x: embedder.encode(x))
    
    X, meta = aggregate_daily(df, embedder)
    if X is None or len(X) < 1:
        past_days = 0
        future_days = 0
    else:
        past_days = len(meta)
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X)
        
        print("Running Mamba State Space Model...")
        encoder = TemporalSSM(X.shape[-1])
        head = LifespanPredictor(X.shape[-1])
        future_days = predict_viral_days(X_scaled, encoder, head)

    print("Generating AI Insight via LLM (based on Top 20 filtered posts)...")
    summary_text = get_llm_summary(df)
    
    display(Markdown(f"### X Trend Report: #{keyword}"))
    display(Markdown(f"**Viral Duration (Past):** {past_days} days"))
    # display(Markdown(f"**Predicted Future Viral Days:** {future_days} days"))
    
    display(Markdown("#### Top Trending Tweets:"))
    display(Markdown(get_trending_posts_display(df, keyword)))
    
    display(Markdown("#### AI Summary:"))
    display(Markdown(summary_text))

In [28]:
# --- CELL 4: EXECUTION ---
keyword = input("Enter hashtag/topic: ")
if keyword:
    run_twitter_trend_demo(keyword)


Analyzing X/Twitter Trend for: 'gta 6'
Fetching Tweets...
Generating COHERENT simulated data for 'gta 6'...
Processing embedding vectors...
Running Mamba State Space Model...
Generating AI Insight via LLM (based on Top 20 filtered posts)...


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### X Trend Report: #gta 6

**Viral Duration (Past):** 6 days

#### Top Trending Tweets:

- [The internet is debating the latest statement about gta 6.](https://x.com/search?q=gta 6) (❤️ 44305 | 💬 4430 | 🔁 11076)
- [Live coverage: Tracking the latest updates on the gta 6 phenomenon.](https://x.com/search?q=gta 6) (❤️ 34854 | 💬 3485 | 🔁 8713)
- [Why gta 6 remains a top priority topic for voters/consumers this week.](https://x.com/search?q=gta 6) (❤️ 34509 | 💬 3450 | 🔁 8627)
- [Opinion: The recent gta 6 updates could change everything we know.](https://x.com/search?q=gta 6) (❤️ 34348 | 💬 3434 | 🔁 8587)
- [Thread: Here is everything you need to know about the gta 6 situation 🧵](https://x.com/search?q=gta 6) (❤️ 23855 | 💬 2385 | 🔁 5963)

#### AI Summary:

Analysts are predicting a significant shift in strategy for gta 6 in 2025 . Major new developments reported on gTa 6 this morning . Sources confirm new timeline for upcoming events . The internet is debating the latest statement about gtian 6 .